# 🎓 Tajweed Rule Analysis

Deep dive into Tajweed rule detection and scoring.

## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'src'))

from src.tajweed_rules import TajweedDetector, TajweedRuleDB, tajweed_score_to_json
from src.preprocess import load_audio_safe, normalize_audio_level
import librosa
import numpy as np
import matplotlib.pyplot as plt
import json

print("✅ Modules loaded")

## Tajweed Rules Database

In [ ]:
# Explore available Tajweed rules
rules = TajweedRuleDB.get_all_rules()

print("📖 Available Tajweed Rules:\n")
for rule_id, rule in rules.items():
    print(f"✨ {rule.rule_name} ({rule.arabic_name})")
    print(f"   ID: {rule.rule_id}")
    print(f"   Description: {rule.description}")
    if rule.duration_min_ms:
        print(f"   Duration: {rule.duration_min_ms}-{rule.duration_max_ms}ms")
    if rule.hafs_specific:
        print(f"   🔹 Hafs-specific")
    if rule.warsh_specific:
        print(f"   🔹 Warsh-specific")
    print()

## Load Sample Audio

In [ ]:
# Load audio sample
sample_path = Path('./data/raw/hafs/001_001_hafs.wav')

if sample_path.exists():
    print(f"📂 Loading: {sample_path.name}")
    audio, sr = load_audio_safe(sample_path, target_sr=16000)
    audio = normalize_audio_level(audio, target_db=-20.0)
    
    print(f"✅ Audio loaded")
    print(f"  Duration: {len(audio) / sr:.2f}s")
    print(f"  Sample rate: {sr}Hz")
else:
    print(f"⚠️  Sample not found: {sample_path}")
    print("Please place sample audio file there first")

## Detect Individual Rules

In [ ]:
if 'audio' in locals():
    detector = TajweedDetector(sample_rate=16000)
    
    print("🔍 Detecting Tajweed Rules...\n")
    
    # Detect each rule
    results = {}
    
    ghunnah = detector.detect_ghunnah(audio)
    results['Ghunnah'] = ghunnah
    print(f"✨ Ghunnah (Nasal Resonance)")
    print(f"   Score: {ghunnah['score']:.1f}/100")
    print(f"   Confidence: {ghunnah['confidence']:.2f}")
    print(f"   Duration: {ghunnah['duration_ms']:.1f}ms")
    print()
    
    idgham = detector.detect_idgham(audio)
    results['Idgham'] = idgham
    print(f"✨ Idgham (Consonant Assimilation)")
    print(f"   Score: {idgham['score']:.1f}/100")
    print(f"   Type: {idgham['type']}")
    print(f"   Confidence: {idgham['confidence']:.2f}")
    print()
    
    imalah = detector.detect_imalah(audio)
    results['Imalah'] = imalah
    print(f"✨ Imalah (Vowel Shift)")
    print(f"   Score: {imalah['score']:.1f}/100")
    print(f"   Formant Shift: {imalah['formant_shift_hz']:.1f}Hz")
    print(f"   Confidence: {imalah['confidence']:.2f}")
    print()
    
    lam_tafkhim = detector.detect_lam_tafkhim(audio)
    results['Lam Tafkhim'] = lam_tafkhim
    print(f"✨ Lam Tafkhim (Emphasis)")
    print(f"   Score: {lam_tafkhim['score']:.1f}/100")
    print(f"   Confidence: {lam_tafkhim['confidence']:.2f}")

## Complete Evaluation

In [ ]:
if 'audio' in locals():
    print("\n📊 Complete Tajweed Evaluation\n")
    print("=" * 60)
    
    score = detector.score_tajweed(
        waveform=audio,
        predicted_qiraat='Hafs',
        expected_qiraat='Hafs',
        qiraat_confidence=0.92,
    )
    
    # Export as JSON
    result_json = tajweed_score_to_json(score)
    print(json.dumps(result_json, indent=2, ensure_ascii=False))
    print("\n" + "=" * 60)

## Visualize Rule Scores

In [ ]:
if 'results' in locals():
    import matplotlib.pyplot as plt
    
    # Extract scores
    rule_names = list(results.keys())
    rule_scores = [results[r]['score'] for r in rule_names]
    confidences = [results[r]['confidence'] for r in rule_names]
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Scores
    colors = ['#2ecc71' if s > 70 else '#f39c12' if s > 50 else '#e74c3c' for s in rule_scores]
    axes[0].barh(rule_names, rule_scores, color=colors)
    axes[0].set_xlabel('Score')
    axes[0].set_title('Tajweed Rule Scores (0-100)')
    axes[0].set_xlim(0, 100)
    for i, v in enumerate(rule_scores):
        axes[0].text(v + 2, i, f'{v:.1f}', va='center')
    
    # Confidences
    axes[1].barh(rule_names, confidences, color='#3498db')
    axes[1].set_xlabel('Confidence')
    axes[1].set_title('Detection Confidence (0-1)')
    axes[1].set_xlim(0, 1)
    for i, v in enumerate(confidences):
        axes[1].text(v + 0.02, i, f'{v:.2f}', va='center')
    
    plt.tight_layout()
    plt.show()

## Hafs vs Warsh Comparison

In [ ]:
print("🔄 Qira'at-Specific Characteristics\n")
print("HAFS RECITATION:")
print("  ✓ Ghunnah: Shorter duration (~30-40ms)")
print("  ✓ Lam Tafkhim: Emphasis on Lam after Damma")
print("  ✓ Imalah: Weak or absent")
print()
print("WARSH RECITATION:")
print("  ✓ Ghunnah: Longer duration (~40-60ms)")
print("  ✓ Imalah: Strong vowel shift toward Ya")
print("  ✓ Idgham: Characteristic nasal type")
print()
print("⚡ Key Differentiators:")
print("  1. Ghunnah duration (most reliable)")
print("  2. Imalah strength (very strong in Warsh)")
print("  3. Lam Tafkhim (Hafs-specific)")

## Batch Analysis

In [ ]:
from pathlib import Path
import pandas as pd

def analyze_directory(directory_path, qiraat_type):
    """Analyze all audio files in a directory"""
    detector = TajweedDetector(sample_rate=16000)
    results = []
    
    audio_files = list(Path(directory_path).glob('*.wav'))
    print(f"Analyzing {len(audio_files)} {qiraat_type} files...")
    
    for i, audio_file in enumerate(audio_files[:5]):  # First 5 files
        try:
            audio, sr = load_audio_safe(audio_file, target_sr=16000)
            audio = normalize_audio_level(audio, target_db=-20.0)
            
            score = detector.score_tajweed(
                audio,
                predicted_qiraat=qiraat_type,
                expected_qiraat=qiraat_type,
                qiraat_confidence=0.90,
            )
            
            results.append({
                'File': audio_file.name,
                'Overall Accuracy': f"{score.overall_accuracy*100:.1f}%",
                'Ghunnah': f"{score.rule_scores.get('ghunnah', 0):.1f}",
                'Idgham': f"{score.rule_scores.get('idgham', 0):.1f}",
                'Imalah': f"{score.rule_scores.get('imalah', 0):.1f}",
            })
        except Exception as e:
            print(f"  ⚠️  Error processing {audio_file.name}: {e}")
    
    return pd.DataFrame(results)

# Analyze if directory exists
hafs_dir = Path('./data/raw/hafs')
if hafs_dir.exists():
    df = analyze_directory(hafs_dir, 'Hafs')
    print("\n📊 Results:\n")
    print(df.to_string(index=False))
else:
    print("⚠️  Hafs directory not found")

## Reference: Islamic Standards

### Ghunnah (الغنة) - Nasal Resonance
- **Definition**: Nasal resonance in Meem (م) or Noon (ن)
- **Source**: Ibn Al-Jazari, Tajweed books
- **Hafs**: ~30-40ms (crisper)
- **Warsh**: ~40-60ms (more extended)

### Idgham (الإدغام) - Consonant Assimilation
- **3 Types**:
  1. Perfect (كامل): No nasal sound
  2. Nasal (بغنة): With Ghunnah
  3. Partial (ناقص): Incomplete assimilation
- **7 Affected Pairs**: ن→{ي, و, م, ن, ل, ر}, م→م

### Imalah (الإمالة) - Vowel Shifting
- **Definition**: Shift Alef (ا) toward Ya (ي) sound
- **Warsh**: Strong (primary characteristic)
- **Hafs**: Weak or absent
- **Detection**: Formant frequency shift

### Lam Tafkhim (تفخيم اللام) - Emphasis
- **Definition**: Emphasis on Lam (ل) after Damma (ُ)
- **Hafs-specific**: Not in other variants
- **Detection**: Lower formant frequencies